# CafeBazaar fake comment detection notebook
Implementation of the "Joint Use of Time Series and Graph Data for Fake Comment Detection in CafeBazaar Dataset" paper: https://ieeexplore.ieee.org/stamp/stamp.jsp?arnumber=11153829

This is the first version of the code that I developed in 2016.
In 2026, I used Copilot to clean up the code and make it more clear but didn't change the logic of the code.

It follows the same idea as the original work: build app-level time-series features from review timing, rating deviation, and weekly review counts, then use an LSTM autoencoder to detect abnormal review patterns.

The output of this pipeline will be an input to the Rev2's implementation. You can find the code fo Rev2 here: https://cs.stanford.edu/~srijan/rev2/


In [ ]:
from pathlib import Path
import math

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

%matplotlib inline

base_dir = Path.cwd()
print(f"Notebook directory: {base_dir}")

# Prefer the already-preprocessed file if present. The original notebooks also used a raw file named
# "review details.csv" in some stages.
for candidate in [base_dir / "review_details_v3", base_dir / "review details.csv"]:
    if candidate.exists():
        data_path = candidate
        break
else:
    raise FileNotFoundError("No review dataset found. Put review_details_v3 or review details.csv in the notebook directory.")

print(f"Using dataset: {data_path}")
df = pd.read_csv(data_path)
print(df.head())
print(f"\nRows: {len(df)}, columns: {list(df.columns)}")


In [ ]:
# Basic EDA from the legacy notebooks
print("Counts of reviews:", len(df))
print("Counts of apps:", df["app_id"].nunique())
print("Counts of users:", df["user_id"].nunique())

if "publish_date" in df.columns:
    df["publish_date"] = pd.to_datetime(df["publish_date"])

# Keep only apps with at least 20 reviews, matching the old filtering strategy
df = df[df.groupby("app_id")["id"].transform("count") >= 20].copy()
df = df.sort_values(["app_id", "publish_date"]).reset_index(drop=True)
print(f"After filtering apps with < 20 reviews: {len(df)} rows across {df['app_id'].nunique()} apps")

# A quick rating histogram
fig, ax = plt.subplots(figsize=(8, 4))
sns.countplot(x=df["rate"], palette="coolwarm", ax=ax)
ax.set_title("Rating distribution")
plt.show()


In [ ]:
# Legacy preprocessing: mark first review of each app and compute inter-review delay
first_ids = df.groupby("app_id")["id"].first()
df["is_first_review"] = df["id"].isin(first_ids.values).astype(int)

# Each row stores the delay from the previous review in hours; first review gets 0
delay = [0.0]
for i in range(1, len(df)):
    previous_ts = df.loc[i - 1, "publish_date"]
    current_ts = df.loc[i, "publish_date"]
    diff_hours = (current_ts - previous_ts).total_seconds() / 3600.0
    delay.append(0.0 if df.loc[i, "is_first_review"] else diff_hours)

df["delay"] = delay

# If the raw dataset does not have a score column, keep it as the app mean rating deviation later.
if "score" not in df.columns:
    df["score"] = df["rate"]

print(df[["id", "app_id", "publish_date", "rate", "delay", "is_first_review"]].head())


In [ ]:
# Convert each application into a weekly time series of: delay, rating deviation, review count

sequence_length = 20
def build_weekly_sequences(df, sequence_length=20):
    sequences = []
    app_ids = []

    for app_id, app_df in df.groupby("app_id"):
        app_df = app_df.sort_values("publish_date").copy()
        if len(app_df) < sequence_length:
            continue

        first_ts = app_df["publish_date"].iloc[0]
        app_df["time_since_first_review"] = (app_df["publish_date"] - first_ts).dt.total_seconds()
        app_df["week_of_review"] = (app_df["time_since_first_review"] / (7 * 24 * 3600)).astype(int)

        weekly = app_df.groupby("week_of_review").agg(
            Delay=("delay", "mean"),
            Stars=("rate", "mean"),
            Count=("id", "count"),
        ).reset_index()

        total_count = weekly["Count"].sum()
        if total_count == 0:
            continue
        weekly["Count"] = weekly["Count"] / total_count

        app_mean = weekly["Stars"].mean()
        weekly["Stars"] = weekly["Stars"] - app_mean

        feature_df = weekly[["Delay", "Stars", "Count"]].copy()
        for col in feature_df.columns:
            vmin = feature_df[col].min()
            vmax = feature_df[col].max()
            if pd.isna(vmin) or pd.isna(vmax) or vmax == vmin:
                feature_df[col] = 0.0
            else:
                feature_df[col] = (feature_df[col] - vmin) / (vmax - vmin)

        values = feature_df.to_numpy(dtype=float)

        # Skip apps that cannot form a full sequence.
        if values.shape[0] < sequence_length:
            continue

        # Exact match: keep this full-length sequence as-is.
        if values.shape[0] == sequence_length:
            sequences.append(values)
            app_ids.append(app_id)
            continue

        # Sliding windows for longer apps.
        for start in range(0, values.shape[0] - sequence_length + 1):
            sequences.append(values[start:start + sequence_length])
            app_ids.append(app_id)

    X = np.stack(sequences, axis=0) if sequences else np.empty((0, sequence_length, 3), dtype=float)
    return X, app_ids

sequence_length = 20
X, app_ids = build_weekly_sequences(df, sequence_length=sequence_length)
print(f"Produced {X.shape[0]} sequences with shape {X.shape}")
print("Example application IDs:", app_ids[:5])
print(f"Produced {X.shape[0]} sequences with shape {X.shape}")
print("Example application IDs:", app_ids[:5])


In [ ]:
# Optional export of app-level CSV files for legacy usage
export_dir = base_dir / "apps"
export_dir.mkdir(exist_ok=True)

for app_id, app_df in df.groupby("app_id"):
    app_df = app_df.sort_values("publish_date").copy()
    first_ts = app_df["publish_date"].iloc[0]
    app_df["time_since_first_review"] = (app_df["publish_date"] - first_ts).dt.total_seconds()
    app_df["week_of_review"] = (app_df["time_since_first_review"] / (7 * 24 * 3600)).astype(int)
    app_df.to_csv(export_dir / str(app_id), index=False)

print(f"Legacy app CSV files written to {export_dir}")


## Sequence autoencoder model

This section consolidates the LSTM autoencoder used in the old code. The goal is to reconstruct normal weekly patterns and flag abnormal sequences by high reconstruction error.


In [ ]:
# Import Keras/TF only when the environment supports it.
try:
    from keras import backend as K
    from keras.layers import Dense, LSTM, RepeatVector, TimeDistributed
    from keras.models import Sequential
except ImportError:  # pragma: no cover
    raise ImportError(
        "This training cell requires TensorFlow/Keras. Install it in the environment before running the model."
    )

if X.shape[0] == 0:
    raise ValueError("No sequences were built from the dataset. Check the review data and filtering conditions.")

# Keep a manageable training set for demonstration purposes; the original notebooks were much larger.
max_samples = min(X.shape[0], 2000)
X_demo = X[:max_samples]

model = Sequential()
model.add(LSTM(64, input_shape=(sequence_length, 3), return_sequences=True))
model.add(LSTM(32, return_sequences=True))
model.add(LSTM(32))
model.add(RepeatVector(sequence_length))
model.add(LSTM(32, return_sequences=True))
model.add(LSTM(64, return_sequences=True))
model.add(TimeDistributed(Dense(3, activation="linear")))
model.compile(optimizer="adam", loss="mse")

model.summary()

# Fit on the same sequence as target; this is an autoencoder setup from the legacy notebook.
model.fit(X_demo, X_demo, epochs=5, batch_size=16, verbose=1)
predicted = model.predict(X_demo, verbose=0)
reconstruction_error = np.mean((X_demo - predicted) ** 2, axis=(1, 2))
print("Reconstruction error statistics:")
print(pd.Series(reconstruction_error).describe())

# Rank the most anomalous sequences
ranked = pd.DataFrame({
    "app_id": [app_ids[i] for i in range(len(reconstruction_error))],
    "reconstruction_error": reconstruction_error
}).sort_values("reconstruction_error", ascending=False)
print(ranked.head(10))


## Interpretation

- `Delay`: time gap between consecutive reviews in hours, often connected to bursty behavior.
- `Stars`: rating deviation from the app's mean rating, capturing abnormal user sentiment.
- `Count`: weekly review volume normalized by the app's total review count.
- High reconstruction error indicates a sequence that does not match the normal pattern learned from the rest of the data.